# Project Overview

# Architecture

# Configuration

In [25]:
from financial_lakehouse.config import (
    DEFAULT_END_DATE,
    DEFAULT_START_DATE,
)
from financial_lakehouse.ingestion import download_market_data

tickers = [
    "AAPL",
    "JPM",
    "XOM",
    # ...
]

# Run Ingestion

In [3]:
market_data = download_market_data(
    tickers=tickers,
    start_date=DEFAULT_START_DATE,
    end_date=DEFAULT_END_DATE,
)

Downloaded AAPL: 2,514 rows
Downloaded JPM: 2,514 rows
Downloaded XOM: 2,514 rows


In [4]:
# inspect raw data
market_data["AAPL"].head()

Price,Date,Adj Close,Close,High,Low,Open,Volume,ticker,download_timestamp
0,2016-01-04,23.688669,26.337500,26.342501,25.500000,25.652500,270597600,AAPL,2026-09-22 15:50:30.637043+00:00
1,2016-01-05,23.095055,25.677500,26.462500,25.602501,26.437500,223164000,AAPL,2026-09-22 15:50:30.637043+00:00
2,2016-01-06,22.643089,25.174999,25.592501,24.967501,25.139999,273829600,AAPL,2026-09-22 15:50:30.637043+00:00
3,2016-01-07,21.687445,24.112499,25.032499,24.107500,24.670000,324377600,AAPL,2026-09-22 15:50:30.637043+00:00
4,2016-01-08,21.802116,24.240000,24.777500,24.190001,24.637501,283192000,AAPL,2026-09-22 15:50:30.637043+00:00


# Inspect Bronze

In [26]:
from financial_lakehouse.bronze import write_ticker_to_bronze

ticker, row_count = write_ticker_to_bronze(
    df=market_data["AAPL"],
    ticker="AAPL",
)

print(f"{ticker}: {row_count:,} rows")

AAPL: 2,514 rows


In [27]:
market_data["AAPL"].columns

Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'ticker',
       'download_timestamp'],
      dtype='str', name='Price')

In [28]:
from financial_lakehouse.bronze import write_market_data_to_bronze

results = write_market_data_to_bronze(market_data)

results[:5]

[('AAPL', 2514), ('JPM', 2514), ('XOM', 2514)]

In [29]:
from pathlib import Path
import pandas as pd

bronze_files = list(
    Path("../data/lakehouse/bronze/ticker=AAPL").rglob("*.parquet")
)

print(f"Partitions: {len(bronze_files)}")
print(bronze_files[:3])

sample = pd.read_parquet(bronze_files[0])
sample.head()

Partitions: 120
[PosixPath('../data/lakehouse/bronze/ticker=AAPL/year=2020/month=03/data.parquet'), PosixPath('../data/lakehouse/bronze/ticker=AAPL/year=2020/month=04/data.parquet'), PosixPath('../data/lakehouse/bronze/ticker=AAPL/year=2020/month=05/data.parquet')]


,Date,Adj Close,Close,High,Low,Open,Volume,download_timestamp
0,2020-03-02,72.071609,74.702499,75.360001,69.430000,70.570000,341397200,2026-09-22 15:50:30.637043+00:00
1,2020-03-03,69.782700,72.330002,76.000000,71.449997,75.917503,319475600,2026-09-22 15:50:30.637043+00:00
2,2020-03-04,73.019501,75.684998,75.849998,73.282501,74.110001,219178400,2026-09-22 15:50:30.637043+00:00
3,2020-03-05,70.650978,73.230003,74.887497,72.852501,73.879997,187572800,2026-09-22 15:50:30.637043+00:00
4,2020-03-06,69.712723,72.257500,72.705002,70.307503,70.500000,226176800,2026-09-22 15:50:30.637043+00:00


# Inspect Silver

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Financial Lakehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 12:48:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [31]:
from financial_lakehouse.config import BRONZE_DIR

bronze_df = spark.read.parquet(str(BRONZE_DIR))

bronze_df.printSchema()
print(f"Bronze rows: {bronze_df.count():,}")

root
 |-- Date: timestamp_ntz (nullable = true)
 |-- Adj Close: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- download_timestamp: timestamp (nullable = true)
 |-- ticker: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

Bronze rows: 7,542


In [32]:
from financial_lakehouse.silver import build_silver

silver_df = build_silver(bronze_df)

silver_df.printSchema()
print(f"Silver rows: {silver_df.count():,}")

root
 |-- Date: date (nullable = true)
 |-- Adj Close: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- download_timestamp: timestamp (nullable = true)
 |-- ticker: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

Silver rows: 7,542


In [33]:
from financial_lakehouse.silver import check_nulls, check_data_quality
null_results = check_nulls(silver_df)
null_results.show(vertical=True)

-RECORD 0-----------------
 Date               | 0   
 Adj Close          | 0   
 Open               | 0   
 High               | 0   
 Low                | 0   
 Close              | 0   
 Volume             | 0   
 download_timestamp | 0   
 ticker             | 0   
 year               | 0   
 month              | 0   



In [34]:
quality_results = check_data_quality(silver_df)
pd.DataFrame(quality_results)

,rule,violations
0,Closing Price > 0,0
1,Volume >= 0,0
2,High >= Low,0


In [38]:
from financial_lakehouse.silver import write_silver
from financial_lakehouse.config import SILVER_DIR

write_silver(silver_df)

print(f"Silver written to: {SILVER_DIR}")

Silver written to: /Users/rnitto/Projects/portfolio/financial-data-lakehouse/data/lakehouse/silver


In [39]:
silver_files = list(
    SILVER_DIR.glob("ticker=AAPL/**/*.parquet")
)

print(f"Partitions: {len(silver_files)}")
print(silver_files[:3])

Partitions: 120
[PosixPath('/Users/rnitto/Projects/portfolio/financial-data-lakehouse/data/lakehouse/silver/ticker=AAPL/year=2020/month=7/part-00000-b3c0d05c-9650-4777-b6b2-83bd53cbcc13.c000.snappy.parquet'), PosixPath('/Users/rnitto/Projects/portfolio/financial-data-lakehouse/data/lakehouse/silver/ticker=AAPL/year=2020/month=9/part-00000-b3c0d05c-9650-4777-b6b2-83bd53cbcc13.c000.snappy.parquet'), PosixPath('/Users/rnitto/Projects/portfolio/financial-data-lakehouse/data/lakehouse/silver/ticker=AAPL/year=2020/month=8/part-00000-b3c0d05c-9650-4777-b6b2-83bd53cbcc13.c000.snappy.parquet')]


In [40]:
silver_check = spark.read.parquet(str(SILVER_DIR))

print(f"Silver rows: {silver_check.count():,}")

silver_check.printSchema()
silver_check.show(5)

Silver rows: 7,542
root
 |-- Date: date (nullable = true)
 |-- Adj Close: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- download_timestamp: timestamp (nullable = true)
 |-- ticker: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+----------+------------------+------------------+------------------+------------------+------------------+--------+--------------------+------+----+-----+
|      Date|         Adj Close|              Open|              High|               Low|             Close|  Volume|  download_timestamp|ticker|year|month|
+----------+------------------+------------------+------------------+------------------+------------------+--------+--------------------+------+----+-----+
|2019-10-25|105.31460571289062|             125.0| 126.6500015258789|            124.7

# Inspect Gold

In [41]:
from financial_lakehouse.gold import build_gold

gold_df = build_gold(silver_df)

print(f"Gold rows: {gold_df.count():,}")

gold_df.printSchema()

Gold rows: 7,542
root
 |-- Date: date (nullable = true)
 |-- Adj Close: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- download_timestamp: timestamp (nullable = true)
 |-- ticker: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- daily_return: double (nullable = true)
 |-- log_return: double (nullable = true)
 |-- price_change: double (nullable = true)
 |-- ma_5: double (nullable = true)
 |-- ma_20: double (nullable = true)
 |-- ma_50: double (nullable = true)
 |-- price_vs_ma5: double (nullable = true)
 |-- price_vs_ma20: double (nullable = true)
 |-- price_vs_ma50: double (nullable = true)
 |-- volatility_5: double (nullable = true)
 |-- volatility_20: double (nullable = true)
 |-- volatility_50: double (nullable = true)
 |-- avg_volume_20: double (nullable = true)
 |--

# Assemble Time-Series Dataset

In [5]:
from financial_lakehouse.config import GOLD_DIR
gold_df = spark.read.parquet(str(GOLD_DIR))

print(f"Gold rows: {gold_df.count():,}")

Gold rows: 7,542


In [7]:
from financial_lakehouse.datasets import (
    build_classification_dataset,
    build_time_series_dataset,
)

time_series_df = build_time_series_dataset(gold_df)

classification_df = build_classification_dataset(gold_df)

In [8]:
print(f"Gold rows:            {gold_df.count():,}")
print(f"Time-series rows:     {time_series_df.count():,}")
print(f"Classification rows:  {classification_df.count():,}")

Gold rows:            7,542
Time-series rows:     7,482
Classification rows:  7,479


In [45]:
time_series_df.printSchema()
time_series_df.show(5)

root
 |-- Date: date (nullable = true)
 |-- ticker: string (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Adj Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- daily_return: double (nullable = true)
 |-- log_return: double (nullable = true)
 |-- price_change: double (nullable = true)
 |-- ma_5: double (nullable = true)
 |-- ma_20: double (nullable = true)
 |-- ma_50: double (nullable = true)
 |-- price_vs_ma5: double (nullable = true)
 |-- price_vs_ma20: double (nullable = true)
 |-- price_vs_ma50: double (nullable = true)
 |-- volatility_5: double (nullable = true)
 |-- volatility_20: double (nullable = true)
 |-- volatility_50: double (nullable = true)
 |-- avg_volume_20: double (nullable = true)
 |-- momentum_20: double (nullable = true)

+----------+------+------------------+------------------+------------------+------------------+----------

# Validate Dataset

In [46]:
classification_df.groupBy("next_day_up").count().orderBy("next_day_up").show()

+-----------+-----+
|next_day_up|count|
+-----------+-----+
|          0| 3569|
|          1| 3910|
+-----------+-----+



In [47]:
classification_df.groupBy("ticker", "next_day_up").count().orderBy(
    "ticker",
    "next_day_up",
).show()

+------+-----------+-----+
|ticker|next_day_up|count|
+------+-----------+-----+
|  AAPL|          0| 1156|
|  AAPL|          1| 1337|
|   JPM|          0| 1190|
|   JPM|          1| 1303|
|   XOM|          0| 1223|
|   XOM|          1| 1270|
+------+-----------+-----+



In [2]:
import importlib
import financial_lakehouse.datasets

importlib.reload(financial_lakehouse.datasets)

<module 'financial_lakehouse.datasets' from '/Users/rnitto/Projects/portfolio/financial-data-lakehouse/src/financial_lakehouse/datasets.py'>

In [57]:
from financial_lakehouse.gold import write_gold

write_gold(gold_df)

print("Gold layer written successfully.")

Gold layer written successfully.


In [9]:
from financial_lakehouse.datasets import (
    write_classification_dataset,
    write_time_series_dataset,
)

write_time_series_dataset(time_series_df)
write_classification_dataset(classification_df)

print("Time-series dataset written successfully.")
print("Classification dataset written successfully.")

Time-series dataset written successfully.
Classification dataset written successfully.


# Explore Time Series

# Key Observations

# Reproducibility & Next Steps

In [10]:
from financial_lakehouse.config import (
    DEFAULT_END_DATE,
    DEFAULT_START_DATE,
)
from financial_lakehouse.pipeline import run_pipeline

run_pipeline(
    spark=spark,
    tickers=["AAPL", "JPM", "XOM"],
    start_date=DEFAULT_START_DATE,
    end_date=DEFAULT_END_DATE,
)

Starting financial market data lakehouse pipeline.
Downloaded AAPL: 2,514 rows
Downloaded JPM: 2,514 rows
Downloaded XOM: 2,514 rows


Pipeline completed successfully.
